# Exercise 3.2: Table Maintenance Procedures

In this exercise, you'll learn how to maintain Apache Iceberg tables:
- **Compact data files**: Consolidate small files into larger ones
- **Compact manifests**: Reduce metadata overhead
- **Expire snapshots**: Clean up old table history
- **Remove orphan files**: Reclaim storage from failed operations

## Learning Objectives
- Identify when maintenance is needed
- Run data file compaction procedures
- Optimize metadata with manifest compaction
- Configure and run snapshot expiration
- Understand orphan file removal
- Monitor table health metrics

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time

spark = SparkSession.builder \
    .appName("TableMaintenance") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.maintenance")
print("Namespace 'maintenance' created!")

## Part 1: Create a Table with Small File Problem

### Create Table and Generate Small Files

In [ ]:
# Create events table
spark.sql("""
CREATE OR REPLACE TABLE polaris.maintenance.events (
    event_id BIGINT,
    event_type STRING,
    user_id STRING,
    event_timestamp TIMESTAMP,
    event_date DATE
)
USING iceberg
PARTITIONED BY (event_date)
TBLPROPERTIES (
    'write.target-file-size-bytes' = '10485760'  -- 10MB target
)
""")

print("Events table created!")

In [ ]:
# Generate many small files (simulating streaming ingestion)
from datetime import datetime, timedelta, date
import random

print("Generating small files (this simulates streaming writes)...")

base_date = date(2024, 1, 1)
event_types = ['click', 'view', 'purchase', 'logout']
users = [f"user_{i}" for i in range(1, 51)]  # 50 users

# Create 50 small commits (100 records each)
for commit_num in range(50):
    events = []
    for i in range(100):
        event_date = base_date + timedelta(days=commit_num % 5)  # Spread across 5 days
        event_ts = datetime.combine(event_date, datetime.min.time()) + timedelta(
            hours=random.randint(0, 23),
            minutes=random.randint(0, 59)
        )
        events.append((
            commit_num * 100 + i,
            random.choice(event_types),
            random.choice(users),
            event_ts,
            event_date
        ))
    
    # Write small batch
    df = spark.createDataFrame(events, 
        ["event_id", "event_type", "user_id", "event_timestamp", "event_date"])
    df.writeTo("polaris.maintenance.events").append()
    
    if (commit_num + 1) % 10 == 0:
        print(f"  Created {commit_num + 1}/50 commits...")

print("\nSmall files created!")

### Assess the Small File Problem

In [ ]:
# Count files and check sizes
files_df = spark.sql("""
SELECT 
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
    ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb,
    ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
FROM polaris.maintenance.events.files
""")

print("File statistics BEFORE compaction:")
files_df.show()

In [ ]:
# Show distribution of file sizes
print("File size distribution:")
spark.sql("""
SELECT 
    ROUND(file_size_in_bytes / 1024 / 1024, 2) as size_mb,
    partition
FROM polaris.maintenance.events.files
ORDER BY size_mb
LIMIT 20
""").show()

In [ ]:
# Count snapshots
snapshot_count = spark.sql("""
SELECT COUNT(*) as count
FROM polaris.maintenance.events.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

**Problem Identified**: Many small files (way below target size) and many snapshots!

## Part 2: Manifest Compaction

### Check Manifest Statistics

In [ ]:
# Count manifests
print("Manifest statistics BEFORE compaction:")
spark.sql("""
SELECT 
    COUNT(*) as manifest_count,
    ROUND(AVG(length) / 1024, 2) as avg_size_kb
FROM polaris.maintenance.events.manifests
""").show()

### Compact Manifests

In [ ]:
# Compact manifests
print("Compacting manifests...")
start = time.time()

spark.sql("""
CALL polaris.system.rewrite_manifests(
    table => 'maintenance.events'
)
""")

elapsed = time.time() - start
print(f"Manifest compaction completed in {elapsed:.2f} seconds")

In [ ]:
# Check manifests after compaction
print("Manifest statistics AFTER compaction:")
spark.sql("""
SELECT 
    COUNT(*) as manifest_count,
    ROUND(AVG(length) / 1024, 2) as avg_size_kb
FROM polaris.maintenance.events.manifests
""").show()

**Result**: Fewer, larger manifest files → faster query planning!

## Part 3: Data File Compaction

### Compact Data Files

In [ ]:
# Compact data files
print("Compacting data files...")
start = time.time()

spark.sql("""
CALL polaris.system.rewrite_data_files(
    table => 'maintenance.events',
    options => map(
        'target-file-size-bytes', '10485760'  -- 10MB
    )
)
""")

elapsed = time.time() - start
print(f"Data file compaction completed in {elapsed:.2f} seconds")

### Compare Before and After

In [ ]:
# Check file statistics after compaction
print("File statistics AFTER compaction:")
spark.sql("""
SELECT 
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    ROUND(MIN(file_size_in_bytes) / 1024 / 1024, 2) as min_size_mb,
    ROUND(MAX(file_size_in_bytes) / 1024 / 1024, 2) as max_size_mb,
    ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 2) as total_size_mb
FROM polaris.maintenance.events.files
""").show()

In [ ]:
# Show file sizes by partition
print("Files per partition after compaction:")
spark.sql("""
SELECT 
    partition,
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb
FROM polaris.maintenance.events.files
GROUP BY partition
ORDER BY partition
""").show()

**Result**: Far fewer files, much closer to target size!

## Part 4: Snapshot Expiration

### View Snapshot History

In [ ]:
# Count snapshots before expiration
before_count = spark.sql("""
SELECT COUNT(*) as count
FROM polaris.maintenance.events.snapshots
""").collect()[0]['count']

print(f"Snapshots BEFORE expiration: {before_count}")

In [ ]:
# View recent snapshots
print("Recent snapshots:")
spark.sql("""
SELECT 
    snapshot_id,
    committed_at,
    operation
FROM polaris.maintenance.events.snapshots
ORDER BY committed_at DESC
LIMIT 10
""").show(truncate=False)

### Expire Old Snapshots

In [ ]:
# Expire snapshots older than 1 hour (for demo purposes)
# In production, you'd typically use days or weeks
print("Expiring old snapshots...")
start = time.time()

spark.sql("""
CALL polaris.system.expire_snapshots(
    table => 'maintenance.events',
    older_than => TIMESTAMP '2024-12-31 23:59:59',
    retain_last => 5
)
""")

elapsed = time.time() - start
print(f"Snapshot expiration completed in {elapsed:.2f} seconds")

In [ ]:
# Count snapshots after expiration
after_count = spark.sql("""
SELECT COUNT(*) as count
FROM polaris.maintenance.events.snapshots
""").collect()[0]['count']

print(f"Snapshots AFTER expiration: {after_count}")
print(f"Removed {before_count - after_count} snapshots")

### Verify Old Files Are Deleted

In [ ]:
# The old small files should now be removed from storage
print("Current files in table:")
spark.sql("""
SELECT COUNT(*) as file_count
FROM polaris.maintenance.events.files
""").show()

print("Note: Expire snapshots also deletes unreferenced data files!")

## Part 5: Orphan File Removal

### Simulate Orphan Files

Orphan files occur when writes fail partway through.

In [ ]:
import boto3
from botocore.client import Config

# Configure MinIO client
s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='password',
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

# Create a fake orphan file
orphan_content = b"This is an orphan file that was never committed"
orphan_key = 'iceberg/maintenance/events/data/orphan-file-12345.parquet'

s3_client.put_object(
    Bucket='warehouse',
    Key=orphan_key,
    Body=orphan_content
)

print(f"Created orphan file: s3://warehouse/{orphan_key}")

### Remove Orphan Files

In [ ]:
# Remove orphan files
# WARNING: This is expensive - it lists all files in the table directory!
print("Removing orphan files...")
print("(This may take a while as it lists all files in storage)")
start = time.time()

spark.sql("""
CALL polaris.system.remove_orphan_files(
    table => 'maintenance.events',
    older_than => TIMESTAMP '1970-01-01 00:00:00'
)
""")

elapsed = time.time() - start
print(f"Orphan file removal completed in {elapsed:.2f} seconds")

In [ ]:
# Verify orphan file was removed
try:
    s3_client.head_object(Bucket='warehouse', Key=orphan_key)
    print("Orphan file still exists (unexpected!)")
except:
    print("Orphan file was successfully removed!")

## Part 6: Monitoring Table Health

### Key Metrics to Monitor

In [ ]:
# Create a table health report
print("=" * 60)
print("TABLE HEALTH REPORT: polaris.maintenance.events")
print("=" * 60)

# File metrics
file_stats = spark.sql("""
SELECT 
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    ROUND(SUM(file_size_in_bytes) / 1024 / 1024 / 1024, 2) as total_size_gb
FROM polaris.maintenance.events.files
""").collect()[0]

print(f"\nData Files:")
print(f"  Count: {file_stats['file_count']}")
print(f"  Average Size: {file_stats['avg_size_mb']} MB")
print(f"  Total Size: {file_stats['total_size_gb']} GB")

# Manifest metrics
manifest_stats = spark.sql("""
SELECT 
    COUNT(*) as manifest_count
FROM polaris.maintenance.events.manifests
""").collect()[0]

print(f"\nManifests:")
print(f"  Count: {manifest_stats['manifest_count']}")

# Snapshot metrics
snapshot_stats = spark.sql("""
SELECT 
    COUNT(*) as snapshot_count,
    MAX(committed_at) as latest_commit
FROM polaris.maintenance.events.snapshots
""").collect()[0]

print(f"\nSnapshots:")
print(f"  Count: {snapshot_stats['snapshot_count']}")
print(f"  Latest: {snapshot_stats['latest_commit']}")

# Record count
record_count = spark.sql("""
SELECT COUNT(*) as count
FROM polaris.maintenance.events
""").collect()[0]['count']

print(f"\nRecords: {record_count:,}")
print("=" * 60)

### Files Per Partition

In [ ]:
# Files per partition (helps identify hot partitions)
print("Files per partition:")
spark.sql("""
SELECT 
    partition,
    COUNT(*) as file_count,
    ROUND(AVG(file_size_in_bytes) / 1024 / 1024, 2) as avg_size_mb,
    SUM(record_count) as total_records
FROM polaris.maintenance.events.files
GROUP BY partition
ORDER BY partition
""").show()

## Maintenance Best Practices

### Recommended Maintenance Schedule

```python
# Daily (for high-volume streaming tables)
- rewrite_manifests()
- expire_snapshots(older_than=7 days, retain_last=100)

# Weekly
- rewrite_data_files() on partitions with >10 files

# Monthly
- remove_orphan_files() (if you've had failures)

# Ad-hoc
- After major data loads
- When query performance degrades
- After schema changes
```

### Warning Signs

Your table needs maintenance if:
- Average file size < 50% of target
- More than 1000 files per partition
- More than 100 manifests
- More than 1000 snapshots
- Query planning takes more than a few seconds

## Key Takeaways

### Maintenance Operations
1. **Manifest compaction**: Reduce metadata overhead, faster planning
2. **Data compaction**: Consolidate small files, faster scans
3. **Snapshot expiration**: Clean history AND delete old files
4. **Orphan removal**: Expensive! Only run when needed

### Order Matters
1. First: Compact manifests (fast, helps planning)
2. Second: Compact data files (slower, creates new snapshots)
3. Third: Expire snapshots (removes old files from storage)
4. Rarely: Remove orphans (only if you have failures)

### Monitoring
- Track file count and sizes
- Monitor snapshot accumulation  
- Watch query planning time
- Check files per partition

### Cost Optimization
- Compaction = compute cost now, query savings later
- Expiration = storage savings immediately
- Orphan removal = expensive, only run when necessary

## Real-World Tips

- **Streaming tables**: Compact daily or even hourly
- **Batch tables**: Weekly compaction usually sufficient
- **Snapshot retention**: 7-30 days typical, depends on use case
- **Orphan removal**: Monthly at most, or after known failures
- **Automation**: Use orchestration tools (Airflow, etc.) for scheduling

## Challenge Exercise

1. Create your own table with a small file problem
2. Calculate the severity metrics
3. Run maintenance in the correct order
4. Measure the improvement
5. Design a maintenance schedule for your use case

## Cleanup

In [ ]:
# Optional: Drop table
# spark.sql("DROP TABLE IF EXISTS polaris.maintenance.events")
# print("Table dropped!")